# CS 171 Chest X-Ray Pneumonia Diagnosis - Full Pipeline

End-to-end Colab/local notebook for the project. In one run it:

1. Clones the repo (Colab) and installs dependencies
2. Downloads the chest X-ray dataset via `kagglehub`
3. Performs a quick EDA (class counts, sample image, class weights)
4. Loads verified checkpoints from `part-2` (or retrains if `RUN_TRAINING=True`)
5. Runs evaluation (`src.evaluate`) for both models on the test split
6. Generates training curves, confusion matrices, and Grad-CAM overlays (`src.interpret`)
7. Displays a final comparison summary

**Toggle training:** set `RUN_TRAINING = True` in the Setup cell to retrain instead of using the saved checkpoints. Default is `False` for fast reproduction.

In [ ]:
"""Setup: clone repo (Colab), install dependencies, resolve project root."""

from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_NAME = "CS-171-Chest-X-Ray-Medical-Diagnosis"
REPO_URL = "https://github.com/jenilkathrotia/CS-171-Chest-X-Ray-Medical-Diagnosis.git"
REPO_BRANCH = "part-3"

RUN_TRAINING = False

IS_COLAB = "google.colab" in sys.modules


def _run(cmd: list[str], check: bool = False) -> int:
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=check).returncode


if IS_COLAB:
    colab_repo = Path("/content") / REPO_NAME
    if not colab_repo.exists():
        _run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(colab_repo)], check=True)
    os.chdir(colab_repo)
    _run(["git", "fetch", "origin", REPO_BRANCH])
    _run(["git", "checkout", REPO_BRANCH])
    _run(["git", "pull", "origin", REPO_BRANCH])


def resolve_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    colab_candidate = Path("/content") / REPO_NAME
    if (colab_candidate / "src").exists():
        return colab_candidate
    raise FileNotFoundError(
        f"Could not locate project root from cwd={start}. Expected a directory containing src/ and notebooks/."
    )


PROJECT_ROOT = resolve_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

if IS_COLAB and (PROJECT_ROOT / "requirements.txt").exists():
    _run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")])

import torch  # noqa: E402

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_TRAINING:", RUN_TRAINING)
print("torch device:", DEVICE)

## 1. Dataset

Downloads the Chest X-Ray Pneumonia dataset via `kagglehub` and resolves the `chest_xray/` root containing `train/`, `val/`, `test/` splits with `NORMAL` and `PNEUMONIA` classes.

In [ ]:
import kagglehub
import pandas as pd

raw_path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
candidates = [Path(raw_path) / "chest_xray", Path(raw_path)]
DATA_DIR = next(p for p in candidates if (p / "train").is_dir())

CLASSES = ["NORMAL", "PNEUMONIA"]
SPLITS = ["train", "val", "test"]


def split_counts(data_dir: Path) -> pd.DataFrame:
    rows = []
    for split in SPLITS:
        row = {"split": split}
        for cls in CLASSES:
            row[cls] = len(list((data_dir / split / cls).glob("*")))
        row["total"] = sum(row[cls] for cls in CLASSES)
        rows.append(row)
    return pd.DataFrame(rows).set_index("split")


print("DATA_DIR:", DATA_DIR)
counts = split_counts(DATA_DIR)
counts

## 2. Quick EDA

Sample image per class, train class distribution, and computed class weights for the training loss.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.datasets import ImageFolder

from src.datasets import compute_class_weights

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, cls in zip(axes, CLASSES):
    sample = next((DATA_DIR / "train" / cls).glob("*"))
    ax.imshow(Image.open(sample), cmap="gray")
    ax.set_title(cls)
    ax.axis("off")
plt.tight_layout()
plt.show()

train_counts = counts.loc["train", CLASSES]
plt.figure(figsize=(5, 3))
train_counts.plot(kind="bar", color=["#4C78A8", "#F58518"])
plt.title("Train Split Class Distribution")
plt.ylabel("Image count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

weights = compute_class_weights(ImageFolder(str(DATA_DIR / "train")))
print("Class weights (NORMAL, PNEUMONIA):", weights.tolist())

## 3. Checkpoints

If `RUN_TRAINING=False` (default), pulls verified `*_best.pt` files (and CSV logs) from `origin/part-2`. If `RUN_TRAINING=True`, retrains both models from scratch.

In [ ]:
CKPT_DIR = PROJECT_ROOT / "results" / "checkpoints"
LOG_DIR = PROJECT_ROOT / "results" / "logs"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

MODELS = ["custom_cnn", "densenet121"]


def fetch_checkpoints_from_part2() -> None:
    _run(["git", "fetch", "origin", "part-2"])
    paths = [
        f"results/checkpoints/{m}_best.pt" for m in MODELS
    ] + [
        f"results/logs/{m}.csv" for m in MODELS
    ]
    _run(["git", "checkout", "origin/part-2", "--", *paths])


if RUN_TRAINING:
    print("[full_pipeline] training both models from scratch...")
    _run([sys.executable, "-m", "src.train", "--model", "custom_cnn",  "--data-dir", str(DATA_DIR), "--epochs", "15"], check=True)
    _run([sys.executable, "-m", "src.train", "--model", "densenet121", "--data-dir", str(DATA_DIR), "--epochs", "10"], check=True)
else:
    missing = [m for m in MODELS if not (CKPT_DIR / f"{m}_best.pt").exists()]
    if missing:
        print(f"[full_pipeline] missing checkpoints {missing} -> pulling from origin/part-2")
        fetch_checkpoints_from_part2()
    else:
        print("[full_pipeline] checkpoints already present; skipping fetch")

for model_name in MODELS:
    ckpt = CKPT_DIR / f"{model_name}_best.pt"
    print(f"  - {ckpt} exists={ckpt.exists()}")

## 4. Evaluation

Runs `src.evaluate` for both models against the test split. Results are written to `results/metrics/{model}_metrics.json` and rendered as a comparison table below.

In [ ]:
import json
from IPython.display import Markdown, display

METRICS_DIR = PROJECT_ROOT / "results" / "metrics"

for model_name in MODELS:
    _run(
        [sys.executable, "-m", "src.evaluate", "--model", model_name, "--data-dir", str(DATA_DIR)],
        check=True,
    )


def load_metrics(model_name: str) -> dict:
    return json.loads((METRICS_DIR / f"{model_name}_metrics.json").read_text(encoding="utf-8"))


rows = []
for model_name in MODELS:
    payload = load_metrics(model_name)
    report = payload["classification_report"]
    rows.append({
        "model": model_name,
        "accuracy": payload["accuracy"],
        "pneumonia_recall": report["PNEUMONIA"]["recall"],
        "pneumonia_precision": report["PNEUMONIA"]["precision"],
        "pneumonia_f1": report["PNEUMONIA"]["f1-score"],
        "normal_recall": report["NORMAL"]["recall"],
        "normal_precision": report["NORMAL"]["precision"],
        "macro_f1": report["macro avg"]["f1-score"],
        "samples": payload["num_samples"],
    })

comparison = pd.DataFrame(rows).set_index("model").round(4)
display(Markdown("### Test-set comparison"))
display(comparison)

## 5. Visualizations

Training curves from `results/logs/*.csv` and confusion matrices from the metrics JSON files just produced.

In [ ]:
from IPython.display import Image

PLOTS_DIR = PROJECT_ROOT / "results" / "plots"
CM_DIR = PROJECT_ROOT / "results" / "confusion_matrices"

_run([sys.executable, "-m", "src.interpret", "plot-logs"], check=True)
for model_name in MODELS:
    _run(
        [sys.executable, "-m", "src.interpret", "confusion-matrix",
         "--metrics-json", str(METRICS_DIR / f"{model_name}_metrics.json")],
        check=True,
    )

display(Markdown("### Training curves"))
for fname in [
    "custom_cnn_training_curves.png",
    "densenet121_training_curves.png",
    "model_comparison_validation_curves.png",
]:
    p = PLOTS_DIR / fname
    if p.exists():
        display(Markdown(f"**{fname}**"))
        display(Image(filename=str(p)))

display(Markdown("### Confusion matrices"))
for model_name in MODELS:
    p = CM_DIR / f"{model_name}_confusion_matrix.png"
    if p.exists():
        display(Markdown(f"**{model_name}**"))
        display(Image(filename=str(p)))

## 6. Grad-CAM Interpretability

Six Grad-CAM overlays per model showing where the network attends when predicting pneumonia.

In [ ]:
GRADCAM_DIR = PROJECT_ROOT / "results" / "gradcam"

for model_name in MODELS:
    _run(
        [sys.executable, "-m", "src.interpret", "gradcam",
         "--model", model_name, "--data-dir", str(DATA_DIR), "--num-examples", "6"],
        check=True,
    )

for model_name in MODELS:
    display(Markdown(f"### {model_name} Grad-CAM"))
    for img_path in sorted(GRADCAM_DIR.glob(f"{model_name}_gradcam_*.png")):
        display(Image(filename=str(img_path)))

## 7. Final Summary

Concise highlights derived from the regenerated metrics. See [`results/metrics.md`](../results/metrics.md) for the full written report.

In [ ]:
best_idx = comparison["accuracy"].idxmax()
worst_idx = comparison["accuracy"].idxmin()
best = comparison.loc[best_idx]
other = comparison.loc[worst_idx]

lines = [
    f"- **Best model:** `{best_idx}` (accuracy {best['accuracy']:.4f}, "
    f"pneumonia recall {best['pneumonia_recall']:.4f}, "
    f"NORMAL precision {best['normal_precision']:.4f}).",
    f"- **Baseline:** `{worst_idx}` (accuracy {other['accuracy']:.4f}, "
    f"pneumonia recall {other['pneumonia_recall']:.4f}).",
    "- Clinical priority is pneumonia sensitivity (catching positives); "
    f"`{best_idx}` is preferred for screening.",
    "- Full report: `results/metrics.md`. "
    "Re-run this notebook with `RUN_TRAINING=True` to retrain from scratch.",
]
display(Markdown("\n".join(lines)))